# A-001 Capstone — AI Agent in Action
## LlamaIndex + GPT-OSS 120B + A-001 Vector Store + One Sensor Tool + Human-Approved Ticket Simulation

This is a **simple capstone notebook** for showing how the ideas from the AI Agents session fit together.

### The story

```text
User question
     |
     v
GPT-OSS 120B  = reasoning brain
     |
     +--------------------+
     |                    |
     v                    v
Sensor Tool          A-001 RAG Tool
(latest vibration)   (existing vector store)
     |                    |
     +---------+----------+
               |
               v
        Agent assessment
               |
               v
      Ticket recommended?
        /             \
      No               Yes
      |                 |
   Answer        PROPOSE JSON ticket
                        |
                        v
                HUMAN APPROVAL
                        |
                        v
                Save simulated ticket
```

### What this notebook deliberately does **not** do

- It does **not** autonomously create a real maintenance ticket.
- It does **not** autonomously update enterprise systems.
- It does **not** treat one sensor value as proof of a failure.
- It does **not** rebuild your existing A-001 document embeddings.

### What it demonstrates

1. **LLM = brain**
2. **Tools = hands**
3. **Structured data = one sensor reading**
4. **Unstructured knowledge = RAG**
5. **Observe evidence before acting**
6. **Propose an action**
7. **Require a human before committing the action**

> All A-001 data and documents are synthetic training material and are not operational engineering guidance.

## 0. Why this capstone is intentionally simple

The objective is to make the agent architecture visible.

We use only one structured sensor signal:

**`vibration_mm_s`**

We do not perform forecasting, anomaly modelling, or deep sensor analytics here.

The notebook asks a simpler question:

> **What is the latest vibration reading, what do the A-001 documents say, and should a human review a proposed ticket?**

In [0]:
# CELL 1 — Install notebook-scoped libraries
#
# We use:
# - LlamaIndex for the persistent vector store + FunctionTool wrappers
# - Databricks embedding endpoint for semantic retrieval
# - MLflow deployment client for GPT-OSS 120B
#
# GPT-OSS is called directly through the Databricks serving client because
# reasoning endpoints can return structured content blocks that some LlamaIndex
# LLM adapters may not flatten into one plain text string.

%pip install -q -U \
    "typing_extensions>=4.15.0,<5" \
    llama-index \
    llama-index-embeddings-databricks \
    mlflow

# Keep this enabled after a fresh install in Databricks.
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# 1. Imports and configuration

The notebook will try to detect your existing ENEC environment automatically.

It supports the two schema names you have been using:

- `workspace.v2_nuclear_enterprise_360`
- `workspace.nuclear_enterprise_360`

It also checks common A-001 persisted vector-store folder names.

In [0]:
# CELL 2 — Imports + simple configuration

import json
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path

from mlflow.deployments import get_deploy_client

from llama_index.core import Settings, StorageContext, load_index_from_storage
from llama_index.core.tools import FunctionTool
from llama_index.embeddings.databricks import DatabricksEmbedding


# ------------------------------------------------------------
# MODELS
# ------------------------------------------------------------

CHAT_MODEL = "databricks-gpt-oss-120b"
EMBED_MODEL = "databricks-qwen3-embedding-0-6b"

TOP_K = 5


# ------------------------------------------------------------
# SIMPLE TRAINING TRIGGER
# ------------------------------------------------------------
#
# This is a SYNTHETIC DEMO threshold for teaching the agent flow.
# It is NOT an engineering limit or operational instruction.

DEMO_VIBRATION_REVIEW_THRESHOLD = 5.1


# ------------------------------------------------------------
# CANDIDATE DATA LOCATIONS
# ------------------------------------------------------------

SENSOR_TABLE_CANDIDATES = [
    "workspace.v2_nuclear_enterprise_360.sensor_readings_a001_1min",
    "workspace.nuclear_enterprise_360.sensor_readings_a001_1min",
]

VECTOR_STORE_CANDIDATES = [
    Path("/Volumes/workspace/v2_nuclear_enterprise_360/training_files/a001_rag_store"),
    Path("/Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store"),
    Path("/Volumes/workspace/v2_nuclear_enterprise_360/training_files/a001_vector_stores"),
    Path("/Volumes/workspace/nuclear_enterprise_360/training_files/a001_vector_stores"),
]

print("Chat model      :", CHAT_MODEL)
print("Embedding model :", EMBED_MODEL)
print("Top K           :", TOP_K)
print("Demo threshold  :", DEMO_VIBRATION_REVIEW_THRESHOLD, "mm/s")

Chat model      : databricks-gpt-oss-120b
Embedding model : databricks-qwen3-embedding-0-6b
Top K           : 5
Demo threshold  : 5.1 mm/s


# 2. Auto-detect the sensor table and existing vector store

This is only setup convenience.

The agent itself will still have just two information tools:

1. **Sensor Tool**
2. **RAG Tool**

In [0]:
# CELL 3 — Environment auto-detection

def find_existing_table(candidates):
    for table_name in candidates:
        try:
            spark.table(table_name).limit(1).collect()
            return table_name
        except Exception:
            pass
    return None


def find_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return None


SENSOR_TABLE = find_existing_table(SENSOR_TABLE_CANDIDATES)
PERSIST_DIR = find_existing_path(VECTOR_STORE_CANDIDATES)

print("Detected sensor table :", SENSOR_TABLE)
print("Detected vector store :", PERSIST_DIR)

if SENSOR_TABLE is None:
    raise FileNotFoundError(
        "Could not find sensor_readings_a001_1min in either expected schema. "
        "Update SENSOR_TABLE_CANDIDATES in CELL 2."
    )

if PERSIST_DIR is None:
    raise FileNotFoundError(
        "Could not find the existing A-001 LlamaIndex vector store. "
        "Update VECTOR_STORE_CANDIDATES in CELL 2."
    )

# Save simulated tickets beside the existing training files.
TICKET_DIR = PERSIST_DIR.parent / "a001_ticket_simulations"
TICKET_DIR.mkdir(parents=True, exist_ok=True)

print("Ticket simulation dir :", TICKET_DIR)

Detected sensor table : workspace.v2_nuclear_enterprise_360.sensor_readings_a001_1min
Detected vector store : /Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store
Ticket simulation dir : /Volumes/workspace/nuclear_enterprise_360/training_files/a001_ticket_simulations


# 3. Configure the existing LlamaIndex vector store

We **do not rebuild the PDFs**.

The expensive document indexing work has already been done.

The only thing required at question time is:

- load the persisted index;
- use the same embedding model to embed the new question;
- retrieve the most relevant chunks.

In [0]:
# CELL 4 — Databricks credentials + embedding model + vector-store reload

API_ROOT = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

API_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

SERVING_ENDPOINT = f"{API_ROOT}/serving-endpoints"

embed_model = DatabricksEmbedding(
    model=EMBED_MODEL,
    api_key=API_TOKEN,
    endpoint=SERVING_ENDPOINT,
)

Settings.embed_model = embed_model

storage_context = StorageContext.from_defaults(
    persist_dir=str(PERSIST_DIR)
)

a001_index = load_index_from_storage(
    storage_context=storage_context,
    embed_model=embed_model,
)

a001_retriever = a001_index.as_retriever(
    similarity_top_k=TOP_K
)

print("Existing A-001 LlamaIndex vector store loaded.")
print("Vector store:", PERSIST_DIR)

Existing A-001 LlamaIndex vector store loaded.
Vector store: /Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store


# 4. GPT-OSS 120B — the reasoning brain

We call the Databricks GPT-OSS endpoint directly.

Why?

Some reasoning models may return structured response blocks rather than one simple string.

The helper below converts common response shapes into plain text so the rest of the notebook remains easy to teach.

In [0]:
# CELL 5 — GPT-OSS helper

dbx_client = get_deploy_client("databricks")


def _flatten_text(value):
    """Turn common model response shapes into readable text."""

    if value is None:
        return ""

    if isinstance(value, str):
        return value

    if isinstance(value, list):
        parts = []
        for item in value:
            parts.append(_flatten_text(item))
        return "\n".join(part for part in parts if part).strip()

    if isinstance(value, dict):
        # Common chat-completion content block.
        if "text" in value and isinstance(value["text"], str):
            return value["text"]

        if "content" in value:
            return _flatten_text(value["content"])

        if "message" in value:
            return _flatten_text(value["message"])

        # Fallback: inspect values.
        parts = [_flatten_text(v) for v in value.values()]
        return "\n".join(part for part in parts if part).strip()

    # Objects returned by SDKs.
    for attr in ("text", "content", "message"):
        if hasattr(value, attr):
            return _flatten_text(getattr(value, attr))

    return str(value)


def gpt_oss_complete(prompt, max_tokens=1200):
    """Call GPT-OSS 120B and return plain text."""

    response = dbx_client.predict(
        endpoint=CHAT_MODEL,
        inputs={
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            "temperature": 0.1,
            "max_tokens": max_tokens,
        },
    )

    # Most OpenAI-compatible Databricks responses contain choices.
    if isinstance(response, dict) and response.get("choices"):
        first = response["choices"][0]

        if isinstance(first, dict):
            message = first.get("message", first)
            text = _flatten_text(message)
            if text:
                return text.strip()

    text = _flatten_text(response)

    if not text:
        raise ValueError("GPT-OSS returned no readable text.")

    return text.strip()


print("GPT-OSS helper ready.")

GPT-OSS helper ready.


# 5. TOOL 1 — One A-001 sensor reading

This tool intentionally stays simple.

It retrieves only:

- timestamp;
- `vibration_mm_s`.

It does **not** perform forecasting or deep time-series analysis.

For a classroom issue simulation, you can optionally pass a synthetic vibration value such as `8.2`.

In [0]:
# CELL 6 — Sensor tool

def latest_a001_vibration(simulated_vibration_mm_s=None):
    """
    Return one A-001 vibration reading.

    Normal mode:
        reads the latest vibration_mm_s from sensor_readings_a001_1min.

    Simulation mode:
        uses the supplied training-only value instead, while clearly labeling
        it as simulated.
    """

    if simulated_vibration_mm_s is not None:
        value = float(simulated_vibration_mm_s)

        return {
            "asset_id": "A-001",
            "reading_timestamp": datetime.now(timezone.utc).isoformat(),
            "vibration_mm_s": value,
            "source": "SIMULATION",
            "demo_threshold_mm_s": DEMO_VIBRATION_REVIEW_THRESHOLD,
            "above_demo_threshold": value > DEMO_VIBRATION_REVIEW_THRESHOLD,
            "important_note": (
                "Synthetic classroom simulation only. "
                "The demo threshold is not an engineering operating limit."
            ),
        }

    row = spark.sql(
        f"""
        SELECT
            reading_timestamp,
            vibration_mm_s
        FROM {SENSOR_TABLE}
        ORDER BY reading_timestamp DESC
        LIMIT 1
        """
    ).first()

    if row is None:
        raise ValueError("No A-001 sensor row was returned.")

    value = float(row["vibration_mm_s"])

    return {
        "asset_id": "A-001",
        "reading_timestamp": str(row["reading_timestamp"]),
        "vibration_mm_s": value,
        "source": SENSOR_TABLE,
        "demo_threshold_mm_s": DEMO_VIBRATION_REVIEW_THRESHOLD,
        "above_demo_threshold": value > DEMO_VIBRATION_REVIEW_THRESHOLD,
        "important_note": (
            "Synthetic training data. "
            "The demo threshold is used only to demonstrate the agent workflow."
        ),
    }


# Show the real latest synthetic reading.
latest_a001_vibration()

{'asset_id': 'A-001',
 'reading_timestamp': '2026-08-25T17:59:00',
 'vibration_mm_s': 5.5646,
 'source': 'workspace.v2_nuclear_enterprise_360.sensor_readings_a001_1min',
 'demo_threshold_mm_s': 5.1,
 'above_demo_threshold': True,
 'important_note': 'Synthetic training data. The demo threshold is used only to demonstrate the agent workflow.'}

# 6. TOOL 2 — A-001 document RAG

This tool searches your existing LlamaIndex vector store.

### Governance principle

A semantically similar document is not automatically authoritative.

By default:

- **APPROVED/current** material may support current guidance;
- **DRAFT** material is not treated as current authority;
- **SUPERSEDED** material is not treated as current authority.

The user can still explicitly ask about draft or historical material.

In [0]:
# CELL 7 — RAG governance helpers

def infer_document_status(source, metadata):
    """Use stored metadata first; fall back to known A-001 filename patterns."""

    status = (
        metadata.get("approval_status")
        or metadata.get("status")
        or ""
    )

    if status:
        return str(status).upper()

    name = str(source).upper()

    if "V3D" in name or "DRAFT" in name:
        return "DRAFT"

    if "V1" in name or "SUPERSEDED" in name:
        return "SUPERSEDED"

    if "V2" in name or "APPROVED" in name:
        return "APPROVED"

    return "UNKNOWN"


def question_requests_history(question):
    q = question.lower()
    keywords = [
        "draft",
        "historical",
        "history",
        "superseded",
        "old version",
        "previous version",
        "compare versions",
    ]
    return any(word in q for word in keywords)

In [0]:
# CELL 8 — A-001 document retrieval tool

def a001_document_evidence(question):
    """Retrieve governed document evidence from the existing A-001 vector store."""

    results = a001_retriever.retrieve(question)

    include_history = question_requests_history(question)

    accepted = []
    rejected = []

    for rank, item in enumerate(results, start=1):
        node = item.node
        metadata = node.metadata or {}

        source = (
            metadata.get("file_name")
            or metadata.get("source")
            or metadata.get("document_id")
            or metadata.get("file_path")
            or "A001 document"
        )

        page = (
            metadata.get("page_label")
            or metadata.get("page_number")
            or metadata.get("page")
            or "unknown"
        )

        status = infer_document_status(source, metadata)

        record = {
            "rank": rank,
            "source": str(source),
            "page": str(page),
            "status": status,
            "similarity": (
                round(float(item.score), 4)
                if item.score is not None
                else None
            ),
            "text": node.get_content().strip(),
        }

        if status in {"DRAFT", "SUPERSEDED"} and not include_history:
            rejected.append(record)
            continue

        accepted.append(record)

    return {
        "question": question,
        "accepted_evidence": accepted,
        "rejected_non_authoritative": rejected,
    }


# Quick RAG test
rag_test = a001_document_evidence(
    "What approved inspection guidance applies to A-001?"
)

print("Accepted chunks:", len(rag_test["accepted_evidence"]))
print("Rejected draft/superseded:", len(rag_test["rejected_non_authoritative"]))

Accepted chunks: 3
Rejected draft/superseded: 2


# 7. Register the tools with LlamaIndex

This is the teaching moment:

```text
Tool 1 = structured fact from the sensor table
Tool 2 = unstructured document knowledge from RAG
```

The ticket-proposal tool is different:

it **prepares** an action, but it does not commit the action.

In [0]:
# CELL 9 — Ticket proposal helper

def make_ticket_proposal(
    title,
    description,
    priority="REVIEW",
    evidence=None,
):
    """
    Create a proposed ticket as a Python dictionary.

    IMPORTANT:
    This does NOT save anything and does NOT create a real ticket.
    Human approval is required later.
    """

    return {
        "ticket_id": f"SIM-{uuid.uuid4().hex[:8].upper()}",
        "asset_id": "A-001",
        "title": title,
        "description": description,
        "priority": priority,
        "status": "PROPOSED - REQUIRES HUMAN APPROVAL",
        "simulation_only": True,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "evidence": evidence or [],
        "human_approval": {
            "approved": False,
            "approved_by": None,
            "approved_at_utc": None,
        },
    }

In [0]:
# CELL 10 — Register LlamaIndex FunctionTools

sensor_tool = FunctionTool.from_defaults(
    fn=latest_a001_vibration,
    name="a001_sensor_reading",
    description=(
        "Read one A-001 vibration value. "
        "Use for the latest structured sensor fact or for an explicitly requested "
        "synthetic training simulation."
    ),
)

rag_tool = FunctionTool.from_defaults(
    fn=a001_document_evidence,
    name="a001_document_rag",
    description=(
        "Retrieve governed A-001 document evidence from the existing LlamaIndex "
        "vector store. Draft and superseded evidence is excluded by default."
    ),
)

ticket_proposal_tool = FunctionTool.from_defaults(
    fn=make_ticket_proposal,
    name="prepare_ticket_proposal",
    description=(
        "Prepare a simulated A-001 ticket proposal in JSON-compatible form. "
        "This tool cannot approve, save, or submit the ticket."
    ),
)

TOOLS = [
    sensor_tool,
    rag_tool,
    ticket_proposal_tool,
]

print("LlamaIndex tools ready:")
for tool in TOOLS:
    print("-", tool.metadata.name)

LlamaIndex tools ready:
- a001_sensor_reading
- a001_document_rag
- prepare_ticket_proposal


# 8. The simple A-001 agent

For teaching, the orchestration is explicit.

We do **not** hide the workflow inside a complicated agent framework.

The agent does:

1. **ACT:** call the sensor tool;
2. **OBSERVE:** inspect the vibration reading;
3. **ACT:** call the RAG tool;
4. **OBSERVE:** inspect governed document evidence;
5. **REASON:** GPT-OSS synthesizes the evidence;
6. **PROPOSE:** if useful, create a ticket proposal;
7. **STOP:** wait for human approval.

That is enough to demonstrate:

**Reason → Act → Observe → Decide**

In [0]:
# CELL 11 — JSON helpers for GPT-OSS

def extract_json_object(text):
    """Best-effort extraction of one JSON object from model output."""

    text = (text or "").strip()

    # Remove markdown fences if the model used them.
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        pass

    # Fallback: take the widest {...} region.
    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError("Could not parse a JSON object from GPT-OSS output.")

In [0]:
# CELL 12 — Main capstone agent

SYSTEM_RULES = """
You are the A-001 enterprise training agent.

Your job is to combine:
1. ONE structured vibration reading.
2. Governed document evidence retrieved by RAG.

Rules:
- Treat all data as synthetic training material.
- Never claim that one vibration reading proves a failure or root cause.
- The 5.1 mm/s threshold is a classroom demo trigger only, not an engineering limit.
- Prefer APPROVED/current documents for current procedure or policy.
- Do not treat DRAFT or SUPERSEDED documents as current authority.
- If evidence is missing or conflicting, say so.
- Never autonomously create, submit, or approve a maintenance ticket.
- You may recommend a ticket PROPOSAL for human review.
- A ticket recommendation must be based on the supplied sensor evidence and/or
  retrieved document evidence.
- Keep the output concise and traceable.
"""


def run_a001_agent(
    question,
    simulated_vibration_mm_s=None,
    verbose=True,
):
    """
    Simple visible A-001 agent.

    Returns:
        {
          sensor,
          rag,
          assessment,
          ticket_proposal
        }
    """

    question = (question or "").strip()

    if not question:
        raise ValueError("Please provide a question.")

    # ==========================================================
    # ACT 1 — SENSOR TOOL
    # ==========================================================

    sensor = latest_a001_vibration(
        simulated_vibration_mm_s=simulated_vibration_mm_s
    )

    if verbose:
        print("=" * 78)
        print("ACT 1 -> SENSOR TOOL")
        print("=" * 78)
        print(json.dumps(sensor, indent=2, default=str))

    # ==========================================================
    # ACT 2 — RAG TOOL
    # ==========================================================

    rag = a001_document_evidence(question)

    if verbose:
        print("\n" + "=" * 78)
        print("ACT 2 -> A-001 RAG TOOL")
        print("=" * 78)

        for item in rag["accepted_evidence"]:
            print(
                f'[{item["rank"]}] '
                f'{item["source"]} | '
                f'page={item["page"]} | '
                f'status={item["status"]} | '
                f'score={item["similarity"]}'
            )

        if rag["rejected_non_authoritative"]:
            print("\nGovernance filtered:")
            for item in rag["rejected_non_authoritative"]:
                print(
                    "-",
                    item["source"],
                    "|",
                    item["status"],
                )

    # Keep prompt context manageable.
    evidence_for_prompt = []

    for item in rag["accepted_evidence"][:TOP_K]:
        evidence_for_prompt.append(
            {
                "source": item["source"],
                "page": item["page"],
                "status": item["status"],
                "text": item["text"][:1800],
            }
        )

    # ==========================================================
    # REASON — GPT-OSS 120B
    # ==========================================================

    prompt = f"""
{SYSTEM_RULES}

USER QUESTION:
{question}

STRUCTURED SENSOR EVIDENCE:
{json.dumps(sensor, indent=2, default=str)}

APPROVED / ACCEPTED DOCUMENT EVIDENCE:
{json.dumps(evidence_for_prompt, indent=2, default=str)}

Return ONE JSON object only with exactly these keys:

{{
  "condition_summary": "short evidence-based summary",
  "issue_detected_for_demo": true_or_false,
  "reason": "why, based only on supplied evidence",
  "document_sources_used": ["source names"],
  "recommended_human_next_step": "short human next step",
  "ticket_recommended": true_or_false,
  "ticket_priority": "LOW | REVIEW | HIGH",
  "ticket_title": "short title or empty string",
  "ticket_description": "short factual description or empty string"
}}

Important:
- issue_detected_for_demo may be true when the synthetic demo threshold is exceeded
  or document evidence supports review.
- Do not diagnose a failure mode unless supplied evidence explicitly establishes it.
- ticket_recommended means PREPARE A PROPOSAL FOR HUMAN REVIEW, not create a real ticket.
- Output JSON only.
"""

    raw_assessment = gpt_oss_complete(
        prompt,
        max_tokens=1000,
    )

    try:
        assessment = extract_json_object(raw_assessment)
    except Exception:
        assessment = {
            "condition_summary": raw_assessment,
            "issue_detected_for_demo": bool(
                sensor.get("above_demo_threshold")
            ),
            "reason": (
                "GPT-OSS response could not be parsed as JSON. "
                "Raw text is preserved in condition_summary."
            ),
            "document_sources_used": [
                item["source"]
                for item in rag["accepted_evidence"][:TOP_K]
            ],
            "recommended_human_next_step": (
                "Review the supplied evidence manually."
            ),
            "ticket_recommended": bool(
                sensor.get("above_demo_threshold")
            ),
            "ticket_priority": "REVIEW",
            "ticket_title": "",
            "ticket_description": "",
        }

    # ==========================================================
    # PROPOSE — NEVER AUTO-SAVE
    # ==========================================================

    ticket_proposal = None

    if bool(assessment.get("ticket_recommended")):
        evidence_refs = [
            {
                "type": "sensor",
                "source": sensor["source"],
                "timestamp": sensor["reading_timestamp"],
                "vibration_mm_s": sensor["vibration_mm_s"],
            }
        ]

        for item in rag["accepted_evidence"][:3]:
            evidence_refs.append(
                {
                    "type": "document",
                    "source": item["source"],
                    "page": item["page"],
                    "status": item["status"],
                }
            )

        ticket_proposal = make_ticket_proposal(
            title=(
                assessment.get("ticket_title")
                or "A-001 condition review"
            ),
            description=(
                assessment.get("ticket_description")
                or assessment.get("condition_summary", "")
            ),
            priority=assessment.get(
                "ticket_priority",
                "REVIEW",
            ),
            evidence=evidence_refs,
        )

    result = {
        "question": question,
        "sensor_observation": sensor,
        "rag_evidence": rag,
        "assessment": assessment,
        "ticket_proposal": ticket_proposal,
    }

    if verbose:
        print("\n" + "=" * 78)
        print("GPT-OSS ASSESSMENT")
        print("=" * 78)
        print(json.dumps(assessment, indent=2, default=str))

        print("\n" + "=" * 78)
        print("TICKET STAGE")
        print("=" * 78)

        if ticket_proposal:
            print("A ticket is PROPOSED only.")
            print("Nothing has been submitted or saved as an approved ticket.")
            print(json.dumps(ticket_proposal, indent=2, default=str))
        else:
            print("No ticket proposal was created.")

    return result

# 9. First test — use the real latest synthetic sensor reading

This reads the most recent `vibration_mm_s` from the one-minute A-001 table.

The agent then searches the vector store and produces an evidence-based assessment.

In [0]:
# CELL 13 — Real synthetic-data test

real_result = run_a001_agent(
    question=(
        "Review the latest A-001 vibration reading. "
        "Use the approved A-001 documents to explain whether a human review is needed."
    ),
    simulated_vibration_mm_s=None,
    verbose=True,
)

ACT 1 -> SENSOR TOOL
{
  "asset_id": "A-001",
  "reading_timestamp": "2026-08-25T17:59:00",
  "vibration_mm_s": 5.5646,
  "source": "workspace.v2_nuclear_enterprise_360.sensor_readings_a001_1min",
  "demo_threshold_mm_s": 5.1,
  "above_demo_threshold": true,
  "important_note": "Synthetic training data. The demo threshold is used only to demonstrate the agent workflow."
}

ACT 2 -> A-001 RAG TOOL
[2] 08_WO-2026-0817_A001_Inspection_Work_Order.pdf | page=1 | status=OPEN | score=0.706
[3] 04_A001-PROC-INS-001-V2_Approved_Inspection_and_PM_Procedure.pdf | page=2 | status=APPROVED | score=0.6907
[4] 07_A001-CMR-2026-08_Condition_Monitoring_Report.pdf | page=1 | status=APPROVED | score=0.6884
[5] 09_CR-2026-0819_A001_Field_Condition_Report.pdf | page=2 | status=UNKNOWN | score=0.6803

Governance filtered:
- 03_A001-PROC-INS-001-V1_Superseded_Inspection_Procedure.pdf | SUPERSEDED

GPT-OSS ASSESSMENT
{
  "condition_summary": "reasoning\nWe need to produce JSON with required fields.\n\nWe have

# 10. Capstone demonstration — simulate an issue

For the live session, use this cell.

We explicitly simulate:

**vibration = 8.2 mm/s**

This mirrors the teaching story:

```text
Thought:
The synthetic vibration trigger is exceeded.

Act:
Read the structured sensor observation.

Observe:
The value is above the demo review threshold.

Act:
Search A-001 documents with RAG.

Observe:
Retrieve approved evidence.

Reason:
GPT-OSS combines both sources.

Action boundary:
Prepare a ticket proposal — but DO NOT submit it.

Human:
Reviews and approves/rejects the proposed JSON ticket.
```

In [0]:
# CELL 14 — Issue simulation

demo_result = run_a001_agent(
    question=(
        "Pump A-001 has an elevated vibration reading. "
        "Review the sensor observation and approved document evidence. "
        "If appropriate, prepare a ticket proposal for a qualified human to review."
    ),
    simulated_vibration_mm_s=8.2,
    verbose=True,
)

proposed_ticket = demo_result["ticket_proposal"]

print("\nPROPOSED TICKET OBJECT")
print(json.dumps(proposed_ticket, indent=2, default=str))

ACT 1 -> SENSOR TOOL
{
  "asset_id": "A-001",
  "reading_timestamp": "2026-09-24T05:59:10.009560+00:00",
  "vibration_mm_s": 8.2,
  "source": "SIMULATION",
  "demo_threshold_mm_s": 5.1,
  "above_demo_threshold": true,
  "important_note": "Synthetic classroom simulation only. The demo threshold is not an engineering operating limit."
}

ACT 2 -> A-001 RAG TOOL
[1] 08_WO-2026-0817_A001_Inspection_Work_Order.pdf | page=1 | status=OPEN | score=0.702
[3] 06_A001-TSG-001_Troubleshooting_and_Diagnostic_Guide.pdf | page=1 | status=APPROVED | score=0.681
[4] 07_A001-CMR-2026-08_Condition_Monitoring_Report.pdf | page=1 | status=APPROVED | score=0.6757
[5] 07_A001-CMR-2026-08_Condition_Monitoring_Report.pdf | page=1 | status=APPROVED | score=0.6688

Governance filtered:
- 03_A001-PROC-INS-001-V1_Superseded_Inspection_Procedure.pdf | SUPERSEDED

GPT-OSS ASSESSMENT
{
  "condition_summary": "Sensor reading shows 8.2\u202fmm/s vibration for Pump A-001, exceeding the demo threshold of 5.1\u202fmm/s. T

# 11. HUMAN-IN-THE-LOOP CONTROL

This is the most important capstone design choice.

The AI agent is **not allowed to submit its own ticket**.

The next function requires an explicit human approval value.

If `approved=False`:

- nothing is saved.

If `approved=True`:

- the notebook saves a **simulated JSON ticket file**;
- the file remains clearly marked as synthetic training material.

In [0]:
# CELL 15 — Human approval gate

def save_simulated_ticket_after_human_approval(
    ticket_proposal,
    approved,
    approved_by,
):
    """
    Save a simulated ticket ONLY after explicit human approval.

    This function is intentionally NOT included in the agent's LlamaIndex tool list.
    The agent cannot call it autonomously.
    """

    if not ticket_proposal:
        return {
            "saved": False,
            "message": "There is no ticket proposal to approve.",
        }

    if approved is not True:
        return {
            "saved": False,
            "message": (
                "Human approval was not granted. "
                "No ticket file was created."
            ),
            "ticket_id": ticket_proposal.get("ticket_id"),
        }

    approved_ticket = dict(ticket_proposal)

    approved_ticket["status"] = (
        "APPROVED FOR SIMULATION - NOT A REAL MAINTENANCE TICKET"
    )

    approved_ticket["human_approval"] = {
        "approved": True,
        "approved_by": str(approved_by),
        "approved_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    file_name = (
        f'{approved_ticket["ticket_id"]}_'
        f'A001_simulated_ticket.json'
    )

    output_path = TICKET_DIR / file_name

    with open(output_path, "w", encoding="utf-8") as handle:
        json.dump(
            approved_ticket,
            handle,
            indent=2,
            default=str,
        )

    return {
        "saved": True,
        "simulation_only": True,
        "ticket_id": approved_ticket["ticket_id"],
        "path": str(output_path),
        "message": (
            "Human approved the training simulation. "
            "A JSON file was saved. "
            "No external ticketing system was contacted."
        ),
    }

## 11A. Show what happens when the human says **NO**

Nothing should be saved.

In [0]:
# CELL 16 — Human rejects / does not approve

not_approved = save_simulated_ticket_after_human_approval(
    ticket_proposal=proposed_ticket,
    approved=False,
    approved_by="Trainer",
)

print(json.dumps(not_approved, indent=2))

{
  "saved": false,
  "message": "Human approval was not granted. No ticket file was created.",
  "ticket_id": "SIM-EF66BE0C"
}


## 11B. Show what happens when the human says **YES**

Run this only when you want to demonstrate the final simulation step.

It saves one JSON file locally in the Databricks training Volume.

It does **not** call ServiceNow, Jira, SAP, Maximo, or any real ticketing API.

In [0]:
# CELL 17 — Human explicitly approves the SIMULATION

approved_result = save_simulated_ticket_after_human_approval(
    ticket_proposal=proposed_ticket,
    approved=True,
    approved_by="ENEC Training Reviewer",
)

print(json.dumps(approved_result, indent=2))

{
  "saved": true,
  "simulation_only": true,
  "ticket_id": "SIM-EF66BE0C",
  "path": "/Volumes/workspace/nuclear_enterprise_360/training_files/a001_ticket_simulations/SIM-EF66BE0C_A001_simulated_ticket.json",
  "message": "Human approved the training simulation. A JSON file was saved. No external ticketing system was contacted."
}


# 12. Inspect the final ticket JSON

This is the final artefact you can show to the class.

It demonstrates how an agent could prepare a structured payload for a downstream ticketing system **without giving the AI permission to submit it autonomously**.

In [0]:
# CELL 18 — Read back the saved simulation file

if approved_result.get("saved"):
    saved_path = Path(approved_result["path"])

    with open(saved_path, "r", encoding="utf-8") as handle:
        saved_ticket_json = json.load(handle)

    print(json.dumps(saved_ticket_json, indent=2))
else:
    print("No approved simulated ticket file exists.")

{
  "ticket_id": "SIM-EF66BE0C",
  "asset_id": "A-001",
  "title": "Review Elevated Vibration on Pump A-001",
  "description": "Synthetic reading of 8.2\u202fmm/s (2026-09-24) exceeds demo threshold. August monitoring report shows a rising trend to 6.6\u202fmm/s. High\u2011priority work order WO-2026-0817 is open for vibration inspection. Recommend human review and execution of the condition inspection procedure.",
  "priority": "HIGH",
  "status": "APPROVED FOR SIMULATION - NOT A REAL MAINTENANCE TICKET",
  "simulation_only": true,
  "created_at_utc": "2026-09-24T05:59:13.922621+00:00",
  "evidence": [
    {
      "type": "sensor",
      "source": "SIMULATION",
      "timestamp": "2026-09-24T05:59:10.009560+00:00",
      "vibration_mm_s": 8.2
    },
    {
      "type": "document",
      "source": "08_WO-2026-0817_A001_Inspection_Work_Order.pdf",
      "page": "1",
      "status": "OPEN"
    },
    {
      "type": "document",
      "source": "06_A001-TSG-001_Troubleshooting_and_Diagnos

# 13. Optional Databricks question box

This lets a learner type a question directly inside the notebook.

By default it uses the real latest synthetic sensor reading.

For the 8.2 mm/s ticket demonstration, use **CELL 14** instead.

In [0]:
# CELL 19 — Optional learner input

dbutils.widgets.text(
    "A001_agent_question",
    (
        "Review A-001 using the latest vibration reading "
        "and the approved document evidence."
    ),
)

user_question = dbutils.widgets.get(
    "A001_agent_question"
)

widget_result = run_a001_agent(
    question=user_question,
    simulated_vibration_mm_s=None,
    verbose=True,
)

ACT 1 -> SENSOR TOOL
{
  "asset_id": "A-001",
  "reading_timestamp": "2026-08-25T17:59:00",
  "vibration_mm_s": 5.5646,
  "source": "workspace.v2_nuclear_enterprise_360.sensor_readings_a001_1min",
  "demo_threshold_mm_s": 5.1,
  "above_demo_threshold": true,
  "important_note": "Synthetic training data. The demo threshold is used only to demonstrate the agent workflow."
}

ACT 2 -> A-001 RAG TOOL
[2] 04_A001-PROC-INS-001-V2_Approved_Inspection_and_PM_Procedure.pdf | page=2 | status=APPROVED | score=0.7018
[3] 08_WO-2026-0817_A001_Inspection_Work_Order.pdf | page=1 | status=OPEN | score=0.6933
[4] 09_CR-2026-0819_A001_Field_Condition_Report.pdf | page=2 | status=UNKNOWN | score=0.6671
[5] 07_A001-CMR-2026-08_Condition_Monitoring_Report.pdf | page=1 | status=APPROVED | score=0.652

Governance filtered:
- 03_A001-PROC-INS-001-V1_Superseded_Inspection_Procedure.pdf | SUPERSEDED

GPT-OSS ASSESSMENT
{
  "condition_summary": "The latest 1\u2011minute vibration reading for A\u2011001 is 5.56\u

# 14. What to explain while presenting

## Brain

**GPT-OSS 120B**

The model interprets the evidence and decides what it means for the training scenario.

---

## Hand 1 — Structured-data tool

**One sensor value: `vibration_mm_s`**

This shows that an agent can call a structured-data source without needing to load the entire database into the LLM.

---

## Hand 2 — RAG tool

**Existing A-001 LlamaIndex vector store**

This gives the agent policies, procedures, inspection reports, work orders and other document evidence.

---

## Governance

Retrieval relevance is not the same as authority.

A DRAFT or SUPERSEDED procedure may look semantically relevant but should not silently become current guidance.

---

## ReAct idea

```text
Reason
  ↓
Act: read sensor
  ↓
Observe
  ↓
Act: retrieve documents
  ↓
Observe
  ↓
Reason again
  ↓
Propose next step
```

---

## Human-in-the-loop

The agent can:

**recommend → prepare → explain**

The human decides whether to:

**approve → reject → modify**

The save/submit capability is deliberately outside the agent's tool list.

# 15. Capstone takeaway

This small application demonstrates the complete progression:

```text
DATA
  ↓
STRUCTURED SENSOR FACT
  +
UNSTRUCTURED ENTERPRISE KNOWLEDGE
  ↓
LLAMAINDEX TOOLS
  ↓
GPT-OSS REASONING
  ↓
EVIDENCE-BASED ASSESSMENT
  ↓
PROPOSED ACTION
  ↓
HUMAN APPROVAL
  ↓
SIMULATED JSON TICKET
```

### The design principle

> **Give the agent enough capability to help — but not enough authority to bypass human review.**

That is a much better enterprise pattern than letting an LLM independently modify operational systems.

In [0]:
# CELL 20 — Gradio app for the A-001 multi-agent demo

import gradio as gr


def agent_interface(question, simulated_vibration, debug_mode):
    """
    Run the A-001 agent and format output for the Gradio UI.

    Returns:
        assessment_text : str  — human-readable assessment
        ticket_json     : str  — JSON of the proposed ticket (or message)
        debug_log       : str  — full verbose trace when debug_mode is on
    """
    import io
    import contextlib

    vib = simulated_vibration if simulated_vibration and simulated_vibration > 0 else None

    debug_buffer = io.StringIO()

    if debug_mode:
        with contextlib.redirect_stdout(debug_buffer):
            result = run_a001_agent(
                question=question,
                simulated_vibration_mm_s=vib,
                verbose=True,
            )
        debug_text = debug_buffer.getvalue()
    else:
        result = run_a001_agent(
            question=question,
            simulated_vibration_mm_s=vib,
            verbose=False,
        )
        debug_text = "_(Enable Debug Mode to see the full agent trace.)_"

    assessment = result["assessment"]
    sensor = result["sensor_observation"]

    assessment_text = f"""### 📊 Sensor Observation

| Field | Value |
|---|---|
| Source | `{sensor.get('source', 'N/A')}` |
| Timestamp | `{sensor.get('reading_timestamp', 'N/A')}` |
| Vibration | **{sensor.get('vibration_mm_s', 'N/A')} mm/s** |
| Above Demo Threshold | `{sensor.get('above_demo_threshold', 'N/A')}` |

---

### 🧠 GPT-OSS Assessment

**Condition Summary:** {assessment.get('condition_summary', 'N/A')}

**Issue Detected (demo):** `{assessment.get('issue_detected_for_demo', 'N/A')}`

**Reason:** {assessment.get('reason', 'N/A')}

**Document Sources Used:** {', '.join(assessment.get('document_sources_used', [])) or 'N/A'}

**Recommended Human Next Step:** {assessment.get('recommended_human_next_step', 'N/A')}

---

### 🎫 Ticket Recommendation

- **Ticket Recommended:** `{assessment.get('ticket_recommended', 'N/A')}`
- **Priority:** `{assessment.get('ticket_priority', 'N/A')}`
- **Title:** {assessment.get('ticket_title', 'N/A') or '—'}
"""

    ticket_proposal = result["ticket_proposal"]

    if ticket_proposal:
        ticket_json = json.dumps(ticket_proposal, indent=2, default=str)
    else:
        ticket_json = "No ticket proposal was created by the agent."

    if debug_mode:
        rag_raw = json.dumps(result['rag_evidence'], indent=2, default=str)
        if len(rag_raw) > 4000:
            rag_raw = rag_raw[:4000] + "\n... (truncated)"
        debug_log = f"""```
{debug_text}
```

**RAG Evidence (raw):**
```json
{rag_raw}
```

**Full Assessment (raw JSON):**
```json
{json.dumps(assessment, indent=2, default=str)}
```
"""
    else:
        debug_log = debug_text

    return assessment_text, ticket_json, debug_log


def approval_interface(ticket_json_str, approved, approved_by):
    """Handle the human-in-the-loop approval gate."""
    if not ticket_json_str or "No ticket proposal" in str(ticket_json_str):
        return "There is no ticket proposal to approve."

    try:
        ticket_proposal = json.loads(ticket_json_str)
    except Exception:
        return "Could not parse the ticket JSON. Please run the agent first."

    result = save_simulated_ticket_after_human_approval(
        ticket_proposal=ticket_proposal,
        approved=approved,
        approved_by=approved_by or "Anonymous",
    )

    return json.dumps(result, indent=2, default=str)


with gr.Blocks(title="A-001 Multi-Agent Demo", theme=gr.themes.Soft()) as app:

    gr.Markdown(
        """
        # 🏭 A-001 Enterprise Multi-Agent Demo

        This Gradio app wraps the capstone notebook's A-001 agent.

        The agent follows a **Reason → Act → Observe → Decide** loop:
        1. **ACT** — call the vibration sensor tool.
        2. **OBSERVE** — inspect the structured reading.
        3. **ACT** — call the RAG document-evidence tool.
        4. **OBSERVE** — inspect governed document evidence.
        5. **REASON** — GPT-OSS synthesizes all evidence.
        6. **PROPOSE** — optionally prepare a ticket proposal.
        7. **STOP** — wait for human approval.

        > ⚠️ The agent can **propose** a ticket but can **never submit one**.
        """
    )

    # ===================== Agent tab =====================
    with gr.Tab("🤖 Agent"):

        with gr.Row():
            with gr.Column(scale=3):
                question_in = gr.Textbox(
                    label="Question for the A-001 agent",
                    value=(
                        "Review A-001 using the latest vibration reading "
                        "and the approved document evidence."
                    ),
                    lines=3,
                )
                simulated_vibration_in = gr.Number(
                    label="Simulated vibration (mm/s)  — leave 0 for real latest reading",
                    value=0,
                )
                debug_toggle = gr.Checkbox(
                    label="🐛 Debug Mode (show full agent trace, raw RAG evidence, raw JSON)",
                    value=True,
                )
                run_btn = gr.Button("▶ Run Agent", variant="primary")

            with gr.Column(scale=5):
                assessment_out = gr.Markdown(label="Assessment")

        with gr.Accordion("🎫 Proposed Ticket JSON", open=False):
            ticket_out = gr.Code(
                label="Ticket Proposal JSON",
                language="json",
                interactive=False,
            )

        with gr.Accordion("🐛 Debug Log", open=False):
            debug_out = gr.Markdown()

        run_btn.click(
            fn=agent_interface,
            inputs=[question_in, simulated_vibration_in, debug_toggle],
            outputs=[assessment_out, ticket_out, debug_out],
        )

    # ===================== Human Approval tab =====================
    with gr.Tab("👤 Human-in-the-Loop"):
        gr.Markdown(
            """
            ## Human Approval Gate

            The AI agent is **not allowed to submit its own ticket**.

            Paste the ticket JSON from the **Agent** tab, then explicitly
            approve or reject.
            """
        )

        ticket_input = gr.Code(
            label="Ticket Proposal JSON (paste from Agent tab)",
            language="json",
            lines=15,
        )
        approved_in = gr.Checkbox(label="✅ Approve (saves a simulated JSON file)", value=False)
        approved_by_in = gr.Textbox(label="Approved by", value="", placeholder="Your name")
        approve_btn = gr.Button("Submit Decision", variant="primary")
        approval_result_out = gr.Code(
            label="Result",
            language="json",
            interactive=False,
        )

        approve_btn.click(
            fn=approval_interface,
            inputs=[ticket_input, approved_in, approved_by_in],
            outputs=[approval_result_out],
        )

    # ===================== Help tab =====================
    with gr.Tab("ℹ️ Help"):
        gr.Markdown(
            """
            ### How to use this demo

            1. **Agent tab** — Type a question, optionally set a simulated
               vibration value (e.g. `8.2` mm/s), and click **Run Agent**.
            2. **Debug Mode** — When enabled, the full verbose trace
               (sensor reading, RAG evidence, rejected documents, raw
               GPT-OSS JSON) is shown in the debug accordion.
            3. **Human-in-the-Loop tab** — Copy the proposed ticket JSON
               into the input box, check **Approve**, enter your name,
               and submit. A simulated JSON file is saved to the
               Databricks Volume only if approved.

            ### Design principle

            > **Give the agent enough capability to help — but not enough
            > authority to bypass human review.**

            ### Simulated values

            | Value | Meaning |
            |---|---|
            | `0` | Use the real latest synthetic sensor reading |
            | `8.2` | Simulated elevated vibration (triggers ticket demo) |
            | `4.0` | Simulated normal vibration (no ticket expected) |
            """
        )

app.launch(prevent_thread_lock=True)

/home/spark-f54bdc72-1c35-44e8-98aa-90/.ipykernel/75/command-7582404189306791-2414050936:123: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="A-001 Multi-Agent Demo", theme=gr.themes.Soft()) as app:


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://4c9fe9bc2ee7c3eac8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-f54bdc72-1c35-44e8-98aa-90c039fd354d/lib/python3.12/site-packages/gradio/queueing.py", line 861, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-f54bdc72-1c35-44e8-98aa-90c039fd354d/lib/python3.12/site-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-f54bdc72-1c35-44e8-98aa-90c039fd354d/lib/python3.12/site-packages/gradio/blocks.py", line 2695, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-f54bdc72-1c35-44e8-98aa-90c039fd354d/lib/python3.12/site-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await a

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7349dbe070f199a3ed.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://4c9fe9bc2ee7c3eac8.gradio.live
